In [21]:
"""
************************************* READ ME *************************************
This code is for averaging 3 frames with the highest Pearson's correlation to the mid slice

The Pearson file looks like this

**************************************************
Image A: mid
Image B: reg-1-1

Pearson's Coefficient:
r=0.208
**************************************************
Image A: mid
Image B: reg-1-2

Pearson's Coefficient:
r=0.219
**************************************************

I want to extract for every frame which slice is best fitted with the mid one.

The Dimension file looks like this
width, height, channels, slices, frames
256 256 1 5 35

Enjoy.
Patrick Cottilli
"""
import sys
import os
import heapq
import shutil

direct = input("Please provide the full path of the directory you want to analyse.\nIt should look like this: User/username/location-of-the-image/Data\n")
fls = []


#Select the files from the directory
for files in os.listdir(direct):
    path = f"{direct}/"
    
    #Copy the first file, creating a new file to merge the two Pearson files
    shutil.copy(path+"/Pearson-mid1.txt", path+"/Pearson-mid.txt")
        
    if 'Dimensions.txt' in files:
        fls.append(path + files)
    if 'Pearson' in files:
        fls.append(path + files)

i=1 #boolean first line
for file in fls:
    if 'Dimensions' not in file:
        continue
    file = open(file)
    for line in file:
        if i:
            i=0
            continue
        line = line.split(' ')
        slices = int(line[-2])
        frames = int(line[-1])

for file in fls:
    if "mid." in file:
        merge = open(file, "a")
    if "mid2" in file:
        src = open(file, "r")

#This creates the new file with all the Pearson values
merge.write("\n")
merge.write(src.read())


wr = "" #string to store the values and then write them at once
dic1 = {} #Dictionary storing the values for the r
for file in fls:
    if "mid." not in file:
        continue
    file = open(file)
    slc = 0
    frm = 1
    temp = {}
    for line in file:
        if "*" in line:
            slc+=1
        if slc == slices+1:
            vals = [] #get maximum r value
            for key in temp:
                vals.append(temp[key])
            if len(vals) != slices:
                sys.exit("Line 70: You are not taking all the values on every frame.")
            selected = heapq.nlargest(3,vals)
            ind1 = vals.index(selected[0])
            ind2 = vals.index(selected[1])
            ind3 = vals.index(selected[2])
            key1 = f"{frm}-{ind1+1}-{ind2+1}-{ind3+1}"
            dic1[key1] = selected
            wr += f"{key1},"
            slc = 1
            frm += 1
            temp = {}

        if "Image B" in line:
            line = line.split(" ")
            image = line[-1]
            temp[image] = 0
            continue

        elif "r=" in line:
            r = line.split("=")
            r = r[-1]
            r = r.split("\n")[0]
            temp[image] = float(r)

#get maximum r value from the last frame
vals = []
for key in temp:
    vals.append(temp[key])

selected = heapq.nlargest(3,vals)
ind1 = vals.index(selected[0])
ind2 = vals.index(selected[1])
ind3 = vals.index(selected[2])
key1 = f"{frm}-{ind1+1}-{ind2+1}-{ind3+1}"
dic1[key1] = selected
wr += f"{key1}"
    
direct = direct.split("/")
direct = "/".join(direct[:-1])
out = open(direct + "/selected-frames.csv", "w")
wr+="\n"
out.write(wr)
out.close()
print("Command finished!")

Please provide the full path of the directory you want to analyse.
It should look like this: User/username/location-of-the-image/Data
 /Users/cottilp/Desktop/try/20230717-Coronal-FFN200-P60-try/Data


Command finished!
